# Protocol 6 — Select & reduce features: redundancy reduction

A discriminant feature set is rarely a *parsimonious* one. A CPP **signature** (Protocol 1) contains hundreds of position-resolved physicochemical features — neighbouring positions and correlated scales within one AAontology category say nearly the same thing — and a raw numeric matrix (e.g. protein language model embedding dimensions) is just as redundant. This protocol shows how to reduce any labelled matrix `(X, y)` to a compact, **non-redundant** subset that still separates the two groups. The headline is a model-free, **CPP-style recipe** — rank by **effect size**, i.e. the adjusted-AUC magnitude `|AUC*|` (the `abs_auc` column of `df_feat`, a feature's group-separating strength; "effect size" is the plain-language synonym used throughout) — then drop features that merely echo an already-kept stronger one — alongside the two other selection mechanisms that coexist in AAanalysis: the model-based `TreeModel.select_features` and `CPP`'s own internal redundancy filter.

This protocol is part of the [Protocols catalog (#35)](https://github.com/breimanntools/aaanalysis/issues/35) and addresses [feature selection & redundancy reduction (#90)](https://github.com/breimanntools/aaanalysis/issues/90). It follows the standard pipeline-chained structure: *When to use it -> Input -> Run -> Output -> How to interpret -> Common mistakes -> Next step.*

## When to use it

Use this protocol once you already have a feature set or a numeric matrix and want a **compact, non-redundant** subset that still separates the **test** group (`label=1`) from the **reference** group (`label=0`). The biological question is: *which* features actually carry the group-separating signal — which TMD/JMD physicochemical features, or which embedding dimensions — once near-duplicates have been removed?

Two complementary stances:

- **Model-free** (univariate): rank each feature on its own separating strength (effect size) and prune correlated echoes. No classifier is fitted; the ranking is the basis. This is the *CPP-style selection on any `(X, y)`* recipe.
- **Model-based**: let a fitted tree report which features it actually relied on, and keep those (`TreeModel.select_features`).

## Input

The input is **any** `(X, y)`: an `(n_samples, n_features)` numeric matrix `X` plus binary integer labels `y` (`1` = test, `0` = reference). Two sources are shown here:

1. A **CPP feature matrix** produced upstream by *Protocol 1 — CPP signature*: `SequenceFeature.feature_matrix` turns the signature `df_feat` + `df_parts` into `X`. This path keeps the per-feature metadata (`feature`, `category`, `abs_auc`, ...).
2. An **arbitrary user matrix** with no CPP metadata — e.g. protein language model embedding dimensions. Here you track surviving *column indices* instead of feature names.

Labels come from the `label` column of `df_seq` (`df_seq["label"].to_list()`), never from `len(df_seq)` (`load_dataset(..., n=N)` returns `2N` rows, N per class).

In [1]:
import aaanalysis as aa
import numpy as np

aa.options["verbose"] = False
aa.options["random_state"] = 42

## Run

### A. Build the CPP feature matrix (upstream, from Protocol 1)

Split the sequences into parts, run `CPP` on the parts to obtain the signature, then materialise the numeric feature matrix `X`. We keep the fixtures small (`n=10` -> 20 sequences) and `n_filter=50` so every cell stays well under the nbmake budget.

In [2]:
df_seq = aa.load_dataset(name="DOM_GSEC", n=10)   # 20 sequences: 10 substrates / 10 non-substrates
labels = df_seq["label"].to_list()

sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
cpp = aa.CPP(df_parts=df_parts)
df_feat = cpp.run(labels=labels, n_filter=50, n_jobs=1)

X = sf.feature_matrix(features=df_feat["feature"], df_parts=df_parts)
X.shape, df_feat[["feature", "abs_auc"]].head()

((20, 50),
                                       feature  abs_auc
 0        JMD_N_TMD_N-Segment(1,10)-ZIMJ680101    0.500
 1  JMD_N_TMD_N-Pattern(C,2,5,8,12)-PALJ810110    0.470
 2          TMD-Pattern(N,1,5,8,11)-PALJ810110    0.470
 3    TMD_C_JMD_C-Pattern(N,4,8,12)-TANS770105    0.470
 4   TMD_C_JMD_C-Pattern(C,8,12,15)-AURR980102    0.465)

### The model-free CPP-style recipe (four steps)

1. **Effect size per feature** — `comp_auc_adjusted` returns the adjusted AUC (`AUC*`) for every column, in `[-0.5, 0.5]`. The sign is *direction* (which group is higher); the **strength** is the magnitude `|AUC*|`.
2. **Sort columns by `|AUC*|` descending** — strongest separators first.
3. **`NumericalFeature.filter_correlation` on the SORTED matrix.** This is **order-dependent**: it walks the columns left-to-right and keeps the **first** of each correlated pair, discarding any later column whose Pearson correlation with an already-kept one exceeds `max_cor`. Feeding it effect-sorted columns is therefore mandatory — this is the **sort-first contract**. On unsorted input it would keep arbitrary representatives.
4. **Take the top-n** of the survivors.

The mask returned by `filter_correlation` can have **fewer** `True` entries than the requested `n`: pruning correlated echoes removes columns, so the kept set is capped at whatever survives — never padded back up.

Note that `X` here is **already** redundancy-reduced: `CPP.run` in section A applied its own per-AAontology-category scale-correlation filter (default `max_cor=0.5`, `max_overlap=0.5`, `check_cat=True`). This model-free pass is therefore *layered* on top of it. It still prunes a lot — even at the looser `max_cor=0.7` — because `filter_correlation` computes Pearson correlation on the actual feature-matrix **columns across all categories**, so it catches **cross-category echoes** that CPP's per-category filter never compares. The two filters see different redundancy, which is why a permissive second pass keeps shrinking an already-0.5-filtered set.

In [3]:
# 1) effect size, 2) effect-sorted column order
auc = aa.comp_auc_adjusted(X=X, labels=labels, n_jobs=1)
order = np.argsort(np.abs(auc), kind="stable")[::-1]  # stable: deterministic tie order

# 3) correlation filter on the EFFECT-SORTED matrix (sort-first contract)
nf = aa.NumericalFeature()
mask = nf.filter_correlation(X=X[:, order], max_cor=0.7)
kept = order[mask]          # original column indices, still in effect order

# 4) top-n of the non-redundant survivors
top_n = kept[:20]

# funnel: kept can be < n_features once correlated echoes are pruned (never padded up)
funnel = {"n_features": X.shape[1], "n_after_filter": int(mask.sum()), "n_top": len(top_n)}
funnel, df_feat.iloc[top_n][["feature", "category", "abs_auc"]].head()

({'n_features': 50, 'n_after_filter': 10, 'n_top': 10},
                                       feature      category  abs_auc
 0        JMD_N_TMD_N-Segment(1,10)-ZIMJ680101      Polarity    0.500
 3    TMD_C_JMD_C-Pattern(N,4,8,12)-TANS770105  Conformation    0.470
 2          TMD-Pattern(N,1,5,8,11)-PALJ810110  Conformation    0.470
 5   TMD_C_JMD_C-Pattern(N,7,10,14)-AURR980102  Conformation    0.465
 10        TMD-Pattern(N,3,7,11,15)-NAKH920108   Composition    0.455)

### B. The same recipe on an arbitrary numeric matrix (embedding analog)

For a raw `(X, y)` with **no** CPP metadata — e.g. protein language model embedding dimensions (the antibody-non-specificity use case from #90) — the identical four steps apply. The only difference: you track surviving *column indices* rather than feature names. Below, columns **0 and 4** carry the separating signal, while columns **1 and 5** are noisier echoes of them: each echo shares its partner's noise (so they stay highly correlated) but carries only half the class shift (so its `|AUC*|` is clearly weaker). Within each correlated pair the sort-first filter keeps the **stronger original** (0 and 4, ranked first by effect size) and drops the **weaker echo** (1 and 5).

In [4]:
rng = np.random.default_rng(42)
Xe = rng.normal(size=(40, 12))
ye = np.array([0, 1] * 20)

# inject a separating signal into two dimensions (columns 0 and 4) ...
Xe[ye == 1, 0] += 1.5
Xe[ye == 1, 4] += 1.0
# ... then add echoes: columns 1 and 5 reuse each partner's noise (-> highly
# correlated) but carry only HALF the class shift (-> clearly weaker effect).
Xe[:, 1] = Xe[:, 0].copy(); Xe[ye == 1, 1] -= 0.75
Xe[:, 5] = Xe[:, 4].copy(); Xe[ye == 1, 5] -= 0.5

auc_e = aa.comp_auc_adjusted(X=Xe, labels=ye, n_jobs=1)
order_e = np.argsort(np.abs(auc_e), kind="stable")[::-1]
mask_e = nf.filter_correlation(X=Xe[:, order_e], max_cor=0.7)
kept_dims = order_e[mask_e]

# stronger originals 0 and 4 survive; their weaker echoes 1 and 5 are dropped
{"order": order_e.tolist(), "kept_dims": kept_dims.tolist()}

{'order': [0, 4, 1, 11, 6, 5, 2, 7, 10, 3, 8, 9],
 'kept_dims': [0, 4, 11, 6, 2, 7, 10, 3, 8, 9]}

### C. Alternative — model-based `TreeModel.select_features`

When you have (or fit) a tree model, you can select by **learned importance** instead of univariate effect size. `TreeModel.fit` computes Monte-Carlo feature importances; `select_features` then row-filters `df_feat` by one `strategy` + one `param` knob (ADR-0023):

- `'top_k'` — keep the `param` (int) features with the highest `feat_importance`.
- `'threshold'` — keep features with `feat_importance >= param` (float).
- `'frequency'` — keep features chosen in at least a `param` fraction of rounds; **requires** `fit(use_rfe=True)` (otherwise every round keeps all features, a no-op `RuntimeWarning`).

A selection that retains **no** features raises `ValueError` rather than returning an empty frame.

In [5]:
tm = aa.TreeModel()
# fit() computes feat_importance regardless of use_rfe; top_k/threshold use it
# directly. Only the frequency strategy needs use_rfe=True.
tm = tm.fit(X, labels=labels)
df_feat_imp = tm.add_feat_importance(df_feat=df_feat)
df_sel = tm.select_features(df_feat=df_feat_imp, strategy="top_k", param=15)

df_sel.shape, df_sel[["feature", "feat_importance"]].head()

((15, 15),
                                       feature  feat_importance
 0        JMD_N_TMD_N-Segment(1,10)-ZIMJ680101            3.206
 1  JMD_N_TMD_N-Pattern(C,2,5,8,12)-PALJ810110            5.296
 2          TMD-Pattern(N,1,5,8,11)-PALJ810110            3.950
 3    TMD_C_JMD_C-Pattern(N,4,8,12)-TANS770105            3.929
 4   TMD_C_JMD_C-Pattern(C,8,12,15)-AURR980102            4.350)

### D. Alternative — CPP's internal redundancy filter: inspecting the funnel

`CPP.run` already performs redundancy reduction as part of the algorithm: it ranks candidates by effect size (`abs_auc`), then prunes by **position overlap** (`max_overlap`) and **correlation** (`max_cor`), optionally **per AAontology category** (`check_cat=True`). Because this greedy filter removes redundant features, `n_final` can be **fewer** than the requested `n_filter` — the requested number is a ceiling, not a guarantee. The cell below **inspects this funnel**: on this tiny fixture the tight `max_cor=0.3` / `max_overlap=0.3` still admit enough features to fill the ceiling, so it makes the *stages* visible rather than a shortfall. (The genuine sub-ceiling shortfall is shown in the model-free path above, where 50 features drop to far fewer.)

The funnel is exposed via `return_stats=True` (also stashed in `df_feat.attrs["last_filter_stats"]` and on `cpp.last_filter_stats_`): `n_candidates` -> `n_after_prefilter` -> `n_after_redundancy` -> `n_final`. A genuine shortfall is surfaced as a `RuntimeWarning` (and the deliberate advisory on tiny fixtures is silenced by `tests/pytest.ini`). To retain more features, raise `max_cor` / `max_overlap` (admitting more redundancy).

In [6]:
df_feat2, stats = cpp.run(
    labels=labels, n_filter=50,
    max_cor=0.3, max_overlap=0.3, check_cat=True,
    return_stats=True, n_jobs=1,
)
# n_final (== len(df_feat2)) can be < the requested n_filter
{**stats, "len_df_feat2": len(df_feat2)}

{'n_candidates': 580140,
 'n_after_prefilter': 29007,
 'n_after_redundancy': 50,
 'n_final': 50,
 'len_df_feat2': 50}

## Output

Each path yields a non-redundant subset, in a shape that matches its basis:

| Path | Returns | Shape note |
| --- | --- | --- |
| Model-free recipe | array of kept column indices (`kept`) / top-n (`top_n`); names via `df_feat.iloc[top_n]` | `len(kept)` can be `< n_features` |
| `TreeModel.select_features` | row-filtered `df_feat` (index reset) | `<= len(df_feat)` rows |
| CPP redundancy filter | `df_feat` already capped at `<= n_filter`, plus the funnel `stats` dict | `n_final <= n_filter` |

The model-free recipe is the only one that runs on a **bare** `(X, y)` with no feature metadata.

## How to interpret

| Mechanism | Basis | Input it needs | Knob | When to use | Gotcha |
| --- | --- | --- | --- | --- | --- |
| `filter_correlation` | univariate, model-free | any `(X, y)`; columns **must be effect-sorted** | `max_cor` | a bare matrix, or you trust effect size over a model | sort-first contract; can return `< n` |
| `TreeModel.select_features` | learned model importance | fitted `TreeModel` + `df_feat` | `strategy` + `param` (ADR-0023) | you have / want a classifier in the loop | `'frequency'` needs `fit(use_rfe=True)`; empty selection raises |
| CPP redundancy filter | `abs_auc` + overlap + correlation | sequences -> `CPP.run` | `n_filter`, `max_cor`, `max_overlap`, `check_cat` | building the signature itself | `n_final` can be `< n_filter` |

**Biological reading:** the surviving features are the *non-redundant determinants* of the test group — the distinct physicochemical patterns (or embedding directions) that separate substrates from non-substrates, with each correlated family represented by its single strongest member rather than a cloud of near-duplicates.

## Common mistakes

- **Calling `filter_correlation` on unsorted columns** — it keeps the *first* of each correlated pair, so unsorted input keeps arbitrary (possibly weak) representatives. Always sort by `|AUC*|` first (the sort-first contract).
- **Expecting exactly `n` features back** — both `filter_correlation` and `CPP.run` can return **fewer** than requested once redundant features are pruned. Raise `max_cor` / `max_overlap` to retain more, or inspect the funnel (`return_stats=True` / `df_feat.attrs["last_filter_stats"]`).
- **Treating `AUC*` sign as strength** — the sign is *direction* (which group is higher); the effect size is the magnitude `|AUC*|`. Sort on the absolute value.
- **Wrong label coding** — `comp_auc_adjusted` needs exactly two integer classes (test=1, reference=0); pass them explicitly via `label_test` / `label_ref` if they differ.
- **Using `strategy='frequency'` without `fit(use_rfe=True)`** — it then keeps every feature (a no-op) and warns.

## Next step

Feed the reduced, non-redundant feature set into **Protocol 7 — Build a classifier**, where the compact subset becomes the input matrix for a `TreeModel` and is evaluated with repeated cross-validation.